# 00 · End-to-end overview

This notebook walks Stages **1–6** (the computational half) of the
*de novo* enzyme-design protocol using the `enzyme_design` toolkit.
The heavy GPU models (RFdiffusion, LigandMPNN, AlphaFold, PLACER) are
**not** executed here — instead we build their validated inputs and
process mock outputs, so the whole logic chain is testable on a laptop.

In [1]:
import sys, os
# make the package importable from the notebooks/ directory
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
EXAMPLES = os.path.join(ROOT, "examples")
print("project root:", ROOT)


project root: /home/user/biofx_python/enzyme_design


## Stage 1 — Theozyme (catalytic constraints)

In [2]:
from enzyme_design.theozyme import parse_cst
theo = parse_cst(open(os.path.join(EXAMPLES, 'theozyme.cst')).read())
print('blocks:', len(theo.blocks))
print('catalytic residues:', theo.catalytic_residues())
print('validation problems:', theo.validate())

blocks: 2
catalytic residues: ['SER', 'HIS']
validation problems: []


## Stage 2 — Contig (what is fixed vs generated)

In [3]:
from enzyme_design.contig import build_contig
contig = build_contig([('A', 84, 87)], flank=(10, 120), total_length=(150, 150))
print('contigs   :', contig.to_contigs_string())
print('motif res :', contig.motif_residues())
print('length    :', contig.length_range(), '-> valid?', contig.validate() == [])

contigs   : 10-120,A84-87,10-120
motif res : ['A84', 'A85', 'A86', 'A87']
length    : (24, 244) -> valid? True


## Stage 3 — RFdiffusionAA command (backbone generation)

In [4]:
from enzyme_design.pipeline import RFdiffusionAAJob
rfd = RFdiffusionAAJob(input_pdb=os.path.join(EXAMPLES, 'active_site.pdb'),
                       contig=contig, ligand='LIG', num_designs=1000)
print(rfd.to_shell())

apptainer run --nv rf_se3_diffusion.sif -u run_inference.py inference.deterministic=True diffuser.T=200 inference.input_pdb=/home/user/biofx_python/enzyme_design/examples/active_site.pdb 'contigmap.contigs=['"'"'10-120,A84-87,10-120'"'"']' 'contigmap.length='"'"'150-150'"'"'' inference.ligand=LIG inference.num_designs=1000 inference.output_prefix=output/run1/sample


## Stage 4 — LigandMPNN command (sequence design, catalytic residues fixed)

In [5]:
from enzyme_design.pipeline import LigandMPNNJob, consistency_check
mpnn = LigandMPNNJob(pdb_path='output/run1/sample_0.pdb',
                     fixed_residues=contig.motif_residues(),
                     number_of_batches=8, pack_side_chains=True)
print(mpnn.to_shell(motif_residues=contig.motif_residues()))
print('cross-check problems:', consistency_check(rfd, mpnn))

python run.py --model_type ligand_mpnn --pdb_path output/run1/sample_0.pdb --fixed_residues 'A84 A85 A86 A87' --number_of_batches 8 --pack_side_chains 1 --out_folder mpnn/run1/
cross-check problems: []


## Stage 5 — Filtering
(a) self-consistency, (b) whole-reaction-coordinate preorganization,
then a combined ranking.

In [6]:
import json
from enzyme_design.metrics import read_metrics_csv
from enzyme_design.preorg import preorg_from_rmsd_samples
from enzyme_design.selection import rank_designs, select_diverse

rows = read_metrics_csv(os.path.join(EXAMPLES, 'af2_metrics.csv'))
data = json.load(open(os.path.join(EXAMPLES, 'preorg_ensembles.json')))
profiles = {d: preorg_from_rmsd_samples(d, v['rmsd_samples'], v.get('contacts'))
            for d, v in data['designs'].items()}
clusters = {'design_0001':'A','design_0006':'A','design_0004':'A',
            'design_0002':'B','design_0005':'B','design_0008':'B',
            'design_0003':'C','design_0007':'C'}
ranked = rank_designs(rows, profiles, clusters)
for r in ranked:
    flag = 'PASS' if r.passes_filters else 'fail'
    print(f'{r.design_id}  [{flag}]  score={r.score:.3f}  cluster={r.cluster}')

design_0006  [PASS]  score=0.010  cluster=A
design_0001  [PASS]  score=0.073  cluster=A
design_0008  [PASS]  score=0.203  cluster=B
design_0002  [PASS]  score=0.262  cluster=B
design_0005  [PASS]  score=0.379  cluster=B
design_0004  [fail]  score=0.480  cluster=A
design_0003  [fail]  score=0.736  cluster=C
design_0007  [fail]  score=0.870  cluster=C


### Stage 6 — Select a diverse experimental panel and register it

In [7]:
from enzyme_design.registry import ConstructRegistry, DesignRecord
panel = select_diverse(ranked, n=3, per_cluster=1)
print('panel:', [p.design_id for p in panel])

reg = ConstructRegistry()
seqs = {r.design_id: 'MAGYSTVKDEFHIKLNPQRWG' for r in panel}  # placeholder seqs
for p in panel:
    reg.add(DesignRecord(p.design_id, protein_seq=seqs[p.design_id],
                         catalytic_residues=contig.motif_residues(),
                         cluster=p.cluster, score=p.score))
print('registry size:', len(reg))
for rec in reg:
    print(rec.design_id, 'warnings:', rec.pre_synthesis_warnings())

panel: ['design_0006', 'design_0008', 'design_0001']
registry size: 3
design_0006 warnings: []
design_0008 warnings: []
design_0001 warnings: []


**Done.** From one active-site specification we produced validated
generation/design commands and a filtered, diversity-aware panel — the
exact hand-off to gene synthesis (Stage 6 → 7).